# Đánh giá câu trước khi chọn lại (pre-ranking)

Đánh giá câu do bộ sinh câu tạo ra bằng **beam search thường** (beam 3), chưa qua bộ chọn lại câu. Đây là điểm
"trước khi chọn lại câu" trong các bảng của bài, và cũng chính là ứng viên đầu tiên trong notebook
`coco2014-eval-reranking.ipynb`.

Khi giải mã, câu được kiểm soát ảo giác bằng **luật chặn cố định** trên logits:
`z'(w) = z(w) − δ · 1[w là từ vật thể COCO, o(w) ∉ E(I)]`, với tập bằng chứng
`E(I) = YOLO-World ∪ {vật thể được ≥ r trong K câu truy hồi nhắc tới}`. Đặt `DELTA = 0` để đánh giá mô hình
không chặn (mô hình cơ sở).

**Cấu hình** (cell CẤU HÌNH): `CHECKPOINT`, `VOCAB_FILE`, `SPLIT`, `DELTA`. Mặc định là cấu hình của bài:
YOLO-World `yolov8l-worldv2` ngưỡng 0,25; K = 5 câu truy hồi; r = 3.

**Độ đo:** BLEU-1..4, METEOR, ROUGE-L, CIDEr (tách từ kiểu `simple` như bài), CHAIRs, CHAIRi. Nhãn đúng của CHAIR
là vật thể trong COCO instances hợp với vật thể nhắc tới trong 5 câu tham chiếu.

**Môi trường:** notebook gọi thẳng code của repo `demo/` (không chép lại), nên số liệu khớp với các script đã dùng
cho bài. Cần Python có `torch`, `transformers`, `ultralytics`, `pycocoevalcap`, `nltk`, và Java cho METEOR.
Trên Kaggle: gắn code repo và dữ liệu làm dataset, sửa các đường dẫn ở cell CẤU HÌNH.

Notebook này được sinh bởi `research/scripts/build_eval_notebooks.py`; sửa ở script đó rồi chạy lại để giữ hai
notebook đồng bộ.


In [ ]:
# ==========================================
# CẤU HÌNH
# ==========================================
import json, os, sys, time
from collections import Counter
from pathlib import Path

# Thư mục repo demo (chứa local_captioner.py, chair_metric.py, ...). Trên Kaggle: gắn code repo làm dataset rồi trỏ vào đó.
DEMO_DIR = Path(r"D:/Documents/BaiTap/KiCuiCung/KLCN/demo")
DATASET_DIR = DEMO_DIR.parent / "dataset"
KARPATHY_JSON = DATASET_DIR / "dataset_coco.json"
INSTANCES_JSON = DATASET_DIR / "instances_val2014.json"
IMAGE_DIR = DATASET_DIR / "val2014" / "val2014"      # ảnh val và test của Karpathy đều thuộc COCO val2014
RETRIEVAL_CACHE = DATASET_DIR / "retrieval_cache"    # kho truy hồi đã mã hóa; chưa có thì notebook tự tạo

# Mô hình cần đánh giá
CHECKPOINT = DEMO_DIR / "restval_dynlogits_contrastive_attnpool_epoch40_model.pth"
VOCAB_FILE = DATASET_DIR / "coco2014_karpathy_restval_dynlogits_contrastive_attnpool_vocab.json"

SPLIT = "test"        # "test" | "val"
START, MAX_IMAGES = 0, None   # ảnh xếp theo tên file như các report của bài; vd MAX_IMAGES = 1000 cho tập con val
DELTA = 3.0           # mức chặn của luật cố định; 0 = không chặn (mô hình cơ sở)

YOLO_MODEL = DEMO_DIR / "yolov8l-worldv2.pt"         # YOLO-World
YOLO_CONF, YOLO_MAX_OBJECTS = 0.25, 20
K_RETR = 5            # số câu truy hồi
MAJ = 3               # ngưỡng đa số r: vật thể được >= r câu truy hồi nhắc tới thì vào tập bằng chứng
MAX_NEW_TOKENS = 40
FORCE_CPU = False     # True: không dùng GPU

OUTPUT = DEMO_DIR / "results" / f"eval_pre_{CHECKPOINT.stem}_d{DELTA:g}_{SPLIT}.json"

In [ ]:
# ==========================================
# NẠP CODE REPO, DỮ LIỆU VÀ NHÃN CHAIR
# ==========================================
if FORCE_CPU:
    import torch
    torch.cuda.is_available = lambda: False   # các module của repo tự chọn thiết bị bằng hàm này
sys.path.insert(0, str(DEMO_DIR))
sys.path.insert(0, str(DEMO_DIR / "research" / "scripts"))
os.environ.setdefault("HF_HUB_OFFLINE", "0")

import numpy as np
import torch
from tqdm.auto import tqdm

import chair_metric
import local_captioner
import vocab as vocab_mod
from evaluate_reranked_quality import compute_metrics
from sretr_experiment import Clip, minmax
from yolo_objects import YoloObjectExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
data = json.loads(KARPATHY_JSON.read_text(encoding="utf-8"))
items = sorted(({"image": im["filename"], "references": [s["raw"] for s in im["sentences"]]}
                for im in data["images"] if im["split"] == SPLIT), key=lambda r: r["image"])
items = items[START: START + MAX_IMAGES] if MAX_IMAGES else items[START:]
paths = [IMAGE_DIR / r["image"] for r in items]
missing = [p.name for p in paths if not p.is_file()]
assert not missing, f"Thiếu {len(missing)} ảnh, vd {missing[:3]}"

OBJECTS, INV = chair_metric.load_synonyms()
fn2id = chair_metric.load_filename_to_cocoid(KARPATHY_JSON)
inst = chair_metric.load_instance_objects_by_cocoid(INSTANCES_JSON, INV)
# Nhãn đúng của CHAIR = vật thể trong COCO instances  U  vật thể nhắc tới trong 5 câu tham chiếu
GT = {r["image"]: chair_metric.ground_truth_from_captions(r["references"], OBJECTS, INV) | inst.get(fn2id.get(r["image"]), set())
      for r in items}


def objs_of(text):
    """Danh sách vật thể COCO (đã quy về tên lớp) mà một câu nhắc tới, như CHAIR."""
    return chair_metric.caption_to_words(text, OBJECTS, INV)[1]


def score(captions, meteor=True, spice=False):
    """BLEU-1..4, METEOR, ROUGE-L, CIDEr (×100), SPICE tùy chọn, CHAIRs/CHAIRi; tách từ kiểu 'simple' như bài."""
    refs = [r["references"] for r in items]
    m = compute_metrics(captions, refs, use_meteor=meteor, use_cider=True, use_spice=spice, tokenizer="simple")
    ch = chair_metric.compute_chair({r["image"]: c for r, c in zip(items, captions)}, GT, OBJECTS, INV)
    out = {k: round(float(v) * (100 if k == "CIDEr" else 1), 2) for k, v in m.items()}
    out.update(CHAIRs=round(ch["CHAIRs"], 2), CHAIRi=round(ch["CHAIRi"], 2),
               mean_len=round(float(np.mean([len(c.split()) for c in captions])), 2))
    return out


print(f"{len(items)} ảnh {SPLIT} | thiết bị {DEVICE} | checkpoint {CHECKPOINT.name} | δ = {DELTA:g}")

In [ ]:
# ==========================================
# BẰNG CHỨNG VẬT THỂ E(I) = YOLO-World  U  {vật thể được >= MAJ trong K câu truy hồi nhắc tới}
# ==========================================
# Kho truy hồi: toàn bộ câu mô tả của tập huấn luyện (train + restval), mã hóa bằng CLIP ViT-B/32, lưu cache.
clip32 = Clip()
caps_f, emb_f = RETRIEVAL_CACHE / "train_captions.json", RETRIEVAL_CACHE / "train_caption_emb_b32.npy"
if caps_f.is_file() and emb_f.is_file():
    BANK_CAPS, BANK_EMB = json.loads(caps_f.read_text(encoding="utf-8")), np.load(emb_f)
else:
    BANK_CAPS = [s["raw"].strip().lower() for im in data["images"] if im["split"] in ("train", "restval") for s in im["sentences"]]
    print(f"Mã hóa kho truy hồi: {len(BANK_CAPS)} câu (chỉ làm một lần)")
    BANK_EMB = clip32.text(BANK_CAPS)
    RETRIEVAL_CACHE.mkdir(parents=True, exist_ok=True)
    caps_f.write_text(json.dumps(BANK_CAPS), encoding="utf-8")
    np.save(emb_f, BANK_EMB)

# K câu gần ảnh nhất theo cosine ảnh-câu (CLIP B/32); kho không chứa câu của tập val/test nên không rò nhãn.
img_emb = clip32.images(paths)
dt = torch.float16 if clip32.device == "cuda" else torch.float32
bank_t = torch.from_numpy(BANK_EMB).to(clip32.device, dt)
NB = np.concatenate([(torch.from_numpy(img_emb[i: i + 256]).to(clip32.device, dt) @ bank_t.T).topk(K_RETR, dim=1).indices.cpu().numpy()
                     for i in range(0, len(img_emb), 256)])
del bank_t

extractor = YoloObjectExtractor(str(YOLO_MODEL), YOLO_CONF, YOLO_MAX_OBJECTS)
vocab_mod.set_object_noun_vocab(extractor.class_names())   # danh từ vật thể cho bằng chứng âm của S_belief
YOLO_OBJS = [extractor.extract(p) for p in tqdm(paths, desc="YOLO-World")]
YOLO_SET = [{INV[o.label] for o in objs if o.confidence >= YOLO_CONF and o.label in INV} for objs in YOLO_OBJS]
NB_COUNTS = [Counter(o for j in idx for o in set(objs_of(BANK_CAPS[j]))) for idx in NB]   # số câu truy hồi nhắc tới mỗi vật thể
EVIDENCE = [y | {o for o, n in c.items() if n >= MAJ} for y, c in zip(YOLO_SET, NB_COUNTS)]
print(f"Trung bình {np.mean([len(e) for e in EVIDENCE]):.2f} vật thể trong E(I) mỗi ảnh")

In [ ]:
# ==========================================
# LUẬT CHẶN CỐ ĐỊNH TRÊN LOGITS:  z'(w) = z(w) - δ · 1[w là từ vật thể COCO, o(w) không thuộc E(I)]
# ==========================================
# Áp ở mọi bước giải mã của cả beam search thường và diverse beam search; từ không phải vật thể giữ nguyên.
IRREGULAR = {"men": "man", "women": "woman", "kids": "kid", "guys": "guy", "mice": "mouse"}
VOCAB = json.loads(VOCAB_FILE.read_text(encoding="utf-8"))
TOK2OBJ = {}
for word, idx in VOCAB.items():
    w = IRREGULAR.get(word) or chair_metric._LEMMATIZER.lemmatize(word, pos="n")
    if w in OBJECTS:
        TOK2OBJ[idx] = INV[w]
print(f"{len(TOK2OBJ)} token trong từ điển ứng với vật thể COCO")

CURRENT_BIAS = None
if not hasattr(local_captioner, "_ORIG_DECODE_STEP"):
    local_captioner._ORIG_DECODE_STEP = local_captioner.ImageCaptioningTransformer.decode_step


def _biased_decode_step(self, memory, sequence):
    logits = local_captioner._ORIG_DECODE_STEP(self, memory, sequence)
    return logits if CURRENT_BIAS is None else logits + CURRENT_BIAS


local_captioner.ImageCaptioningTransformer.decode_step = _biased_decode_step


def rule_bias(evidence):
    if not DELTA:
        return None
    bias = torch.zeros(len(VOCAB))
    for tid, obj in TOK2OBJ.items():
        if obj not in evidence:
            bias[tid] = -DELTA
    return bias.to(DEVICE)

In [ ]:
# ==========================================
# SINH CÂU: beam search thường (beam 3), có luật chặn
# ==========================================
captioner = local_captioner.LocalCheckpointCaptioner(checkpoint_path=CHECKPOINT, vocab_path=VOCAB_FILE,
                                                     device=DEVICE, max_length=MAX_NEW_TOKENS)
CAPTIONS = []
t0 = time.time()
for i, path in enumerate(tqdm(paths, desc="Sinh câu")):
    CURRENT_BIAS = rule_bias(EVIDENCE[i])
    text, _, _ = captioner.generate_diverse(image_path=path, max_candidates=1, num_beams=3,
                                            num_beam_groups=1, diversity_penalty=0.0)[0]
    CAPTIONS.append(text)
CURRENT_BIAS = None
print(f"{(time.time() - t0) / len(paths):.2f} giây mỗi ảnh")

In [ ]:
# ==========================================
# TÍNH ĐIỂM VÀ LƯU
# ==========================================
METRICS = score(CAPTIONS)
print(json.dumps(METRICS, ensure_ascii=False, indent=1))

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.write_text(json.dumps({
    "checkpoint": str(CHECKPOINT), "vocab": str(VOCAB_FILE), "split": SPLIT, "start": START, "images": len(items),
    "delta": DELTA, "yolo_model": YOLO_MODEL.name, "yolo_conf": YOLO_CONF, "k_retr": K_RETR, "maj": MAJ,
    "metrics": METRICS,
    "rows": [{"image": r["image"], "caption": c, "references": r["references"], "evidence": sorted(e)}
             for r, c, e in zip(items, CAPTIONS, EVIDENCE)],
}, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Đã lưu {OUTPUT}")